In [1]:
# Get the sqlite database path
#| echo: false

from pathlib import Path

import pandas as pd
from great_tables import GT
from great_tables import html
from sqlalchemy import create_engine
from sqlalchemy import text, bindparam

from nhs_waiting_lists import (
    __app_name__,
)
from nhs_waiting_lists.constants import LARGE_ACUTE_PROVIDER_CODES, TOTAL_ONLY_TREATMENT_CODES
from nhs_waiting_lists.constants import proj_db_path, DB_FILE
from nhs_waiting_lists.utils.xdg import XDGBasedir

project_root = Path(XDGBasedir.get_data_dir(__app_name__))

DB_PATH = project_root / proj_db_path / DB_FILE
FILES_DIR = project_root / "files"

engine = create_engine(f"sqlite:///{DB_PATH}")

# Get the 23 large acute trust provider codes, identified by the ranking table csv
PROVIDER_CODES = LARGE_ACUTE_PROVIDER_CODES

# Get the C_999 meta-treatment code which aggregates all other treatment codes
TREATMENT_CODES = TOTAL_ONLY_TREATMENT_CODES
# TREATMENT_CODES = ALL_TREATMENT_CODES

#
# provider_code = "RAJ"
# treatment_code = "C_110"

In [2]:
#| echo: false
#| output: false

## Retrieve all the provider rtt summary data

start_period = "2024-01"
end_period = "2024-12"

consolidated_query = text("""
                          SELECT i.period,
                                 i.provider,
                                 i.treatment,
                                 i.untreated       AS untreated,
                                 i.new_periods     AS new_periods,
                                 i.incomplete,
                                 i.incomplete_prev AS incomplete_prev,
                                 i.completed       AS completed,
                                 i.total_treatable  AS total_pathways
                          FROM consolidated AS i
                                   JOIN provider AS p ON i.provider = p.provider
                          WHERE i.provider IN :provider_codes
                            AND i.treatment IN :treatment_codes
                            AND i.period >= :start_period
                            AND i.period <= :end_period
                            AND subtype = 'Acute - Large'
                          ORDER BY i.provider ASC, i.period ASC; \
                          """).bindparams(
    bindparam('provider_codes', expanding=True),
    bindparam('treatment_codes', expanding=True),
    bindparam('start_period', expanding=False),
    bindparam('end_period', expanding=False)
)

consolidated_df = pd.read_sql(consolidated_query, engine, params={
    'provider_codes': PROVIDER_CODES,
    'treatment_codes': TREATMENT_CODES,
    'start_period': start_period,
    'end_period': end_period},
                              )  # type: ignore[arg-type]

consolidated_df["period"] = pd.to_datetime(consolidated_df["period"], errors="coerce")

# valid = (consolidated_df['incomplete_prev'] > 100) & (consolidated_df['new_periods'] > 30)
# consolidated_df = consolidated_df.loc[valid].copy()



OperationalError: (sqlite3.OperationalError) no such column: subtype
[SQL: 
                          SELECT i.period,
                                 i.provider,
                                 i.treatment,
                                 i.untreated       AS untreated,
                                 i.new_periods     AS new_periods,
                                 i.incomplete,
                                 i.incomplete_prev AS incomplete_prev,
                                 i.completed       AS completed,
                                 i.total_treatable  AS total_pathways
                          FROM consolidated AS i
                                   JOIN provider AS p ON i.provider = p.provider
                          WHERE i.provider IN (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                            AND i.treatment IN (?)
                            AND i.period >= ?
                            AND i.period <= ?
                

In [ ]:
#| echo: false
#| output: false

other_trusts = consolidated_df.query("provider != 'RAJ'")

other_trusts

## Establishing a Large Acute Trust baseline

As part of the NHS Oversight Framework, NHS England publishes an acute trust league table, which identifies 23 trusts classified as Large Acute Trusts. [@nhs_england_nhs_2025] The Referral to Treatment (RTT) dataset contains monthly submissions for each of these trusts [@nhs_england_referral_2025], segmented by treatment specialty and identified by treatment code. This includes the meta-code C_999, which aggregates activity across all specialties.

Each monthly RTT submission consists of three sections. Part 1 reports completed RTT pathways (clock-stops), subdivided into admitted pathways (Part 1a) and non-admitted pathways (Part 1b). In this analysis, these are collectively referred to as *completed pathways*. Part 2 reports the number of incomplete RTT pathways remaining at the end of the reporting period, referred to here as *incomplete*. Part 3 reports the number of new RTT referrals received during the reporting period, referred to here as *new pathways*.

These quantities imply a simple accounting identity. Absent any unreported changes, the number of incomplete pathways at the end of a reporting period should equal the number of incomplete pathways at the start of the period, plus new pathways, minus completed pathways. Deviations from this identity are calculated as:

$$
\text{UnreportedRemovals}_t =
\text{Incomplete}_t -
\left(
\text{Incomplete}_{t-1}
+
\text{NewPathways}_t
-
\text{Completed}_t
\right)
$$

A negative value of this residual indicates a net reduction in incomplete pathways that is not explained by reported referrals or completions in the published dataset.


### Methods

Monthly RTT reports were consolidated into a single table and aggregated by provider using the C_999 treatment meta-code. For each period, this yields totals for observed incomplete pathways, expected incomplete pathways implied by reported flows, and the absolute residual between the two. These aggregates are then used to calculate the total number of pathways available to treat and the implied percentage net loss attributable to unreported removals.

In this analysis, values drawn from the publicly released RTT CSV files are referred to as “reported” figures, reflecting their provenance as trust-submitted data published by NHS England. Where these figures enter the accounting identity, they are treated as the “observed” quantities in the statistical sense.


In [ ]:
# Echo the base df before applying style sheets
#| echo: false
#| output: false

other_result: pd.DataFrame = other_trusts.groupby(['period'])[
    ['incomplete_prev', 'new_periods', 'completed', "total_treatable", 'untreated', "incomplete"]].sum().reset_index()

other_result["unexplained"] = other_result["untreated"]

other_result["unexplained_pct"] = other_result["untreated"] / other_result["total_pathways"]



### Large Acute Trusts baseline (excluding MSEFT)

Table 1 summarises monthly reported incomplete pathways, expected incomplete pathways implied by reported flows, and the resulting residual for the 22 Large Acute Trusts excluding Mid and South Essex NHS Foundation Trust (MSEFT) during calendar year 2024.

In [ ]:
#| echo: false
#| output: true

(other_result[
     [
         "period",
         "incomplete",
         "incomplete_expected",
         "unexplained",
         "total_treatable",
         "unexplained_pct",
     ]]
 .style.relabel_index(
    [
        "",
        'Observed<br>Incomplete<br>Pathways',
        'Expected<br>Incomplete<br>Pathways',
        'Unreported<br>Change',
        'Treatable<br>Pathways',
        'Unreported<br>%'
    ], axis=1).hide(axis='index')
 .format(precision=3, thousands=",", decimal=".")
 .format('{:.2%}', subset=["unexplained_pct"])
 .format(lambda v: v.strftime("%Y-%m"), subset=["period"])
 )

In [ ]:
#| echo: false
#| output: true

display_other_summary_df = pd.DataFrame({
    'Metric': ['Mean', 'Total'],
    'actual_pathways': [other_result["actual_pathways"].mean(), None],
    'expected_pathways': [other_result["expected_pathways"].mean(), None],
    'unexplained': [other_result["unexplained"].mean(), other_result["unexplained"].sum()],
    'total_pathways': [other_result["total_pathways"].mean(), None],
    'unexplained_pct': [
        other_result['untreated'].sum() / (other_result['incomplete_prev'].sum() + other_result['new_periods'].sum()),
        None],
})

display_other_summary_gt = (
    GT(
        display_other_summary_df)
    .fmt_number(columns=[
        'actual_pathways',
        'expected_pathways',
        'unexplained',
        'total_pathways',
    ], decimals=0)
    .fmt_percent(columns=['unexplained_pct'], decimals=2)
    .cols_label(
        Metric=html("<nbsp/>"),
        actual_pathways=html("<center>Observed<br>Incomplete<br>Pathways</center>"),
        expected_pathways=html("Expected<br>Incomplete<br>Pathways"),
        unexplained=html("Unreported<br>Change"),
        total_pathways=html('Treatable<br>Pathways'),
        unexplained_pct=html("Unreported<br>%"),
    )
)

display_other_summary_gt


Across this group, the mean unreported net reduction in incomplete pathways was approximately 2.83% per month. This figure is consistent with the approximately 3% unreported removal rate previously reported across all provider types for the period April 2023 to March 2025 [@watson_why_2025].

In absolute terms, these 22 Large Acute Trusts collectively recorded approximately 566,000 unreported pathway removals during 2024.


In [ ]:
#| echo: false
#| output: false

raj = consolidated_df.query("provider == 'RAJ'")

raj_result: pd.DataFrame = raj.groupby(['period'])[
    ['incomplete_prev', 'new_periods', 'treated', 'untreated', "incomplete"]].sum().reset_index()

raj_result["total_pathways"] = raj_result["incomplete_prev"] + raj_result["new_periods"]
raj_result["unexplained"] = raj_result["incomplete"] - (
        raj_result["incomplete_prev"] + raj_result["new_periods"] - raj_result["treated"])
raj_result["unexplained_pct"] = raj_result["untreated"] / raj_result["total_pathways"]

raj_result["expected_pathways"] = raj_result["incomplete_prev"] + raj_result["new_periods"] - raj_result["treated"]
raj_result["actual_pathways"] = raj_result["incomplete"]

raj_result[["period", "actual_pathways", "expected_pathways", "unexplained", "total_pathways", "unexplained_pct"]]


## Mid and South Essex NHS Foundation Trust

Applying the same methodology to Mid and South Essex NHS Foundation Trust over the same reporting period produces materially different results, as shown in Table 3.


In [ ]:
#| echo: false
#| output: true

(raj_result[
    [
        "period",
        "actual_pathways",
        "expected_pathways",
        "unexplained",
        "total_pathways",
        "unexplained_pct",
    ]].style.relabel_index(
    [
        "",
        'Observed<br>Incomplete<br>Pathways',
        'Expected<br>Incomplete<br>Pathways',
        'Unreported<br>Change',
        'Treatable<br>Pathways',
        'Unreported<br>%'
    ], axis=1).hide(axis='index')
.format(precision=3, thousands=",", decimal=".")
.format('{:.2%}', subset=["unexplained_pct"])
.format(lambda v: v.strftime("%Y-%m"), subset=["period"])
.set_table_styles([
    {'selector': 'th.col_heading', 'props': 'text-align: center;'},
    # {'selector': 'th.col_heading.level0', 'props': 'font-size: 1.5em;'},
    # {'selector': 'td', 'props': 'text-align: center; font-weight: bold;'},
], overwrite=False)
)

In [ ]:
#| echo: false
#| output: true

display_mseft_summary_df = pd.DataFrame({
    'Metric': ['Mean', 'Total'],
    'actual_pathways': [raj_result["actual_pathways"].mean(), None],
    'expected_pathways': [raj_result["expected_pathways"].mean(), None],
    'unexplained': [raj_result["unexplained"].mean(), raj_result["unexplained"].sum()],
    'total_pathways': [raj_result["total_pathways"].mean(), None],
    'unexplained_pct': [raj['untreated'].sum() / (raj['incomplete_prev'].sum() + raj['new_periods'].sum()), None],
})

display_mseft_summary_gt = (
    GT(
        display_mseft_summary_df)
    .fmt_number(columns=['actual_pathways', 'expected_pathways', 'unexplained', 'total_pathways'], decimals=0)
    .fmt_percent(columns=['unexplained_pct'], decimals=2)
    .cols_label(
        Metric=html("<nbsp/>"),
        actual_pathways=html("<center>Observed<br>Incomplete<br>Pathways</center>"),
        expected_pathways=html("Expected<br>Incomplete<br>Pathways"),
        unexplained=html("Unreported<br>Change"),
        total_pathways=html('Treatable<br>Pathways'),
        unexplained_pct=html("Unreported<br>%"),
    )
)

display_mseft_summary_gt

Across 2024, MSEFT exhibits a mean unreported removal rate of 8.67% of incomplete pathways per month—approximately three times the Large Acute Trust baseline (8.67% vs 2.83%). In absolute terms, this corresponds to 215,408 unreported pathway removals during the year.

Although MSEFT represents only one of the 23 Large Acute Trusts included in this analysis, its unreported removals account for approximately 27.5% of the total unreported removals observed across all Large Acute Trusts in 2024.


In [ ]:
#| echo: false
#| output: false

"{:.3%}".format(raj['untreated'].sum() / (raj['incomplete_prev'].sum() + raj['new_periods'].sum()))



## Excess unreported removals

To control for national effects such as validation exercises, seasonal pressures, or system-wide disruptions (e.g. winter capacity constraints), a period-by-period comparison was conducted between MSEFT and the Large Acute Trust baseline excluding MSEFT.

For each month, the excess unreported removal rate was calculated as the difference between MSEFT’s unreported percentage loss and the corresponding baseline percentage loss among other Large Acute Trusts. Applying this differential to MSEFT’s total pathways yields an estimate of excess unreported removals attributable to deviation from the baseline.



In [ ]:
#| echo: false
#| output: false

comparison_df = (
    raj_result.add_prefix("raj_")
    .merge(other_result.add_prefix("other_"), left_on="raj_period", right_on="other_period")
)
comparison_df

In [ ]:
#| echo: false
#| output: false

comparison_df["excess_pct"] = comparison_df["raj_unexplained_pct"] - comparison_df["other_unexplained_pct"]
comparison_df["excess_unexplained"] = comparison_df["excess_pct"] * comparison_df["raj_expected_pathways"]

comparison_df[[
    "raj_period",
    "raj_unexplained",
    "raj_unexplained_pct",
    "other_unexplained_pct",
    "excess_pct",
    "excess_unexplained"
]]


In [ ]:
#| echo: false
#| output: true

summary_display_df = comparison_df[[
    "raj_period",
    "raj_unexplained",
    "raj_unexplained_pct",
    "other_unexplained_pct",
    "excess_pct",
    "excess_unexplained"
]]

# Style it
comparison_styled = (
    summary_display_df
    .style
    .hide(axis='index')
    .relabel_index(
        [
            "Period",
            'MSEFT<br>Unreported<br>Change',
            'MSEFT<br>Unreported<br>%',
            'Other<br>Unreported<br>%',
            'Excess<br>Unreported<br>%',
            'Excess <br>Unreported<br>Loss'
        ], axis=1).hide(axis='index')
    .format(
        {
            'raj_period': lambda x: x.strftime('%Y-%m') if isinstance(x, pd.Timestamp) else x,
            'raj_unexplained': '{:,.0f}',
            'raj_unexplained_pct': '{:,.2%}',
            'other_unexplained_pct': '{:,.2%}',
            'excess_pct': '{:,.2%}',
            'excess_unexplained': '{:<,.2f}',
        })
)

# comparison_styled = comparison_styled.set_table_styles([
#     {'selector': 'th.col_heading', 'props': 'text-align: center;'},
#     {'selector': 'th', 'props': 'text-align: center;'},
#     {'selector': 'td', 'props': 'text-align: right; font-weight: bold;'},
# ])

comparison_styled


In [ ]:
#| echo: false
#| output: true

display_comparison_summary_df = pd.DataFrame(
    {
        'Metric': ['Mean', 'Total'],
        'raj_unexplained': [comparison_df["raj_unexplained"].mean(), comparison_df['raj_unexplained'].sum()],
        'raj_unexplained_pct': [comparison_df["raj_unexplained_pct"].mean(), None],
        'other_unexplained_pct': [comparison_df["other_unexplained_pct"].mean(), None],
        'excess_pct': [comparison_df["excess_pct"].mean(), None],
        'excess_unexplained': [comparison_df['excess_unexplained'].mean(), comparison_df['excess_unexplained'].sum()],
    })

display_comparison_summary_gt = (
    GT(
        display_comparison_summary_df)
    .fmt_number(columns=['raj_unexplained', 'excess_unexplained'], decimals=0)
    .fmt_percent(columns=['raj_unexplained_pct', 'other_unexplained_pct', 'excess_pct'],
                 decimals=2)
    .cols_label(
        Metric=html("<nbsp/>"),
        raj_unexplained=html("MSEFT<br>Unreported<br>Change"),
        raj_unexplained_pct=html("MSEFT<br>Unreported<br>%"),
        other_unexplained_pct=html("Other<br>Unreported<br>%"),
        excess_pct=html('Excess<br>Unreported<br>%'),
        excess_unexplained=html("Excess<br>Unreported<br>Loss"),
    )
)

display_comparison_summary_gt


Under this comparison, MSEFT recorded 127,974 unreported pathway removals in excess of what would be expected given the observed Large Acute Trust baseline in 2024.

In [ ]:
#| echo: false
#| output: false

comparison_df_value: float = comparison_df[["excess_unexplained"]].sum().item()
print(f"{comparison_df_value:,.0f}")

## Interpretation and implications

Individual RTT pathway removals may be clinically, administratively, or operationally justified, and this analysis does not assess the validity of individual clock-stop decisions. However, the scale, persistence, and concentration of unreported removals observed at MSEFT—particularly when compared with otherwise similar Large Acute Trusts over the same period—raise questions about the consistency and transparency of RTT waiting list management practices.

Further analysis is planned to examine pathway-level mechanisms and alternative explanatory hypotheses.

At this scale, even a small proportion of affected pathways corresponding to unresolved or abandoned care would translate into significant clinical risk for patients who were referred precisely because investigation or treatment was deemed necessary. Nonetheless, the magnitude of the discrepancies identified here is sufficient, even at this preliminary stage, to warrant closer scrutiny.



## References